# HotpotQA Cross-Encoder Evaluation

这个 notebook 复用现有的 HotpotQA span 扫描缓存，只针对输入的 span 读取相关文本并做 cross-encoder 预测。

- 不生成 embedding。
- 候选定义来自 `search_wikidata(span, limit=5, include_detailed_description=True, detailed_description_sentences=3, drop_missing_detailed_description=True)`。
- 如果 `search_wikidata` 没有返回候选定义，直接报错并停止。
- 输出展示原始文本上下文以及预测分数最高的定义。


In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from sentence_transformers import CrossEncoder

from text_processing import normalize_text
from wikidata_utils import fetch_detailed_descriptions_for_entities


In [3]:
HOTPOT_SCAN_STORE_PATH = Path("hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl")
DEFAULT_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
DEFAULT_BATCH_SIZE = 32


def load_hotpot_scan_store(scan_store_path: Path = HOTPOT_SCAN_STORE_PATH):
    if not scan_store_path.exists():
        raise FileNotFoundError(
            f"HotpotQA scan store not found at {scan_store_path}. Please prepare the scan cache first."
        )

    with scan_store_path.open("rb") as handle:
        store = pickle.load(handle)

    print(f"Loaded scan-only store from {scan_store_path}")
    print(store["stats"])
    return store


embedding_store = load_hotpot_scan_store()


Loaded scan-only store from hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl
{'num_documents': 66581, 'num_unique_terms': 634624, 'num_phrase_occurrences': 1007780, 'num_token_occurrences': 584999, 'num_total_occurrences': 1592779}


In [4]:
def lookup_records(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def extract_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = max(
        cleaned_text.rfind(".", 0, start_char),
        cleaned_text.rfind("!", 0, start_char),
        cleaned_text.rfind("?", 0, start_char),
    )
    right_candidates = [
        cleaned_text.find(".", end_char),
        cleaned_text.find("!", end_char),
        cleaned_text.find("?", end_char),
    ]
    right_candidates = [idx for idx in right_candidates if idx != -1]

    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = len(cleaned_text) if not right_candidates else min(right_candidates) + 1
    context_raw = cleaned_text[context_start:context_end]

    if not context_raw.strip():
        context_start = max(0, start_char - 120)
        context_end = min(len(cleaned_text), end_char + 120)
        context_raw = cleaned_text[context_start:context_end]

    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - context_start - left_trim
    local_end = end_char - context_start - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def build_hotpot_prompt(record, query_text, mark_target=False, left_marker="[TGT]", right_marker="[/TGT]"):
    context_info = extract_sentence_context(record["cleaned_text"], record["span"])
    context_text = context_info["context_text"]
    local_start, local_end = context_info["local_span"]
    prompt_context = context_text

    if mark_target:
        prompt_context = (
            f"{context_text[:local_start]}{left_marker} {context_text[local_start:local_end]} {right_marker}{context_text[local_end:]}"
        )

    prompt_text = (
        f"Sentence: {prompt_context}\n"
        f"Target word: {query_text.strip()}\n\n"
        f'Question: What does "{query_text.strip()}" mean in this sentence?'
    )

    return {
        "context_text": context_text,
        "matched_text": context_text[local_start:local_end],
        "local_span": (local_start, local_end),
        "prompt_text": prompt_text,
    }


In [5]:
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"
DEFAULT_HEADERS = {
    "User-Agent": "WikidataExplorerNotebook/1.0 (https://www.wikidata.org/)"
}


def _safe_get_json(url, params=None, headers=None, timeout=30):
    try:
        response = requests.get(url, params=params, headers=headers or DEFAULT_HEADERS, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        print(f"Request failed: {exc}")
        return {}
    except ValueError as exc:
        print(f"Invalid JSON response: {exc}")
        return {}


def _coerce_aliases_for_search(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return ", ".join(str(v) for v in value if v is not None)
    if isinstance(value, str):
        return value
    return str(value)


def _series_casefold_equals(series, target):
    target_casefold = str(target).casefold()
    return series.map(lambda value: str(value).casefold() == target_casefold if value is not None else False)

def _series_non_empty_mask(series):
    return series.map(lambda value: bool(str(value).strip()) if value is not None else False)

def _series_exclude_name_descriptions(series):
    blocked_phrases = ("family name", "given name")
    return series.map(
        lambda value: not any(phrase in str(value).casefold() for phrase in blocked_phrases)
        if value is not None else True
    )

def search_wikidata(
    term,
    language="en",
    limit=10,
    exact_match_text=False,
    include_detailed_description=False,
    drop_missing_detailed_description=False,
    detailed_description_sentences=3,
    filter_name=True,
):
    if not isinstance(term, str) or not term.strip():
        print("Please provide a non-empty search term.")
        columns = ["id", "label", "description", "match_text", "aliases", "concepturi"]
        if include_detailed_description:
            columns.insert(3, "detailed_description")
        return pd.DataFrame(columns=columns)

    normalized_term = term.strip()

    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": language,
        "uselang": language,
        "search": normalized_term,
        "limit": int(limit),
    }

    data = _safe_get_json(WIKIDATA_API_URL, params=params)
    items = data.get("search", []) if isinstance(data, dict) else []

    rows = []
    for item in items:
        match = item.get("match") if isinstance(item, dict) else None
        match_text = ""
        if isinstance(match, dict):
            match_text = match.get("text", "")

        row = {
            "id": item.get("id", ""),
            "label": item.get("label", ""),
            "description": item.get("description", ""),
            "match_text": match_text,
            "aliases": _coerce_aliases_for_search(item.get("aliases")),
            "concepturi": item.get("concepturi", ""),
        }
        rows.append(row)

    df = pd.DataFrame(rows, columns=["id", "label", "description", "match_text", "aliases", "concepturi"])

    if filter_name and not df.empty:
        df = df[_series_exclude_name_descriptions(df["description"])].reset_index(drop=True)

    if exact_match_text and not df.empty:
        df = df[_series_casefold_equals(df["match_text"], normalized_term)].reset_index(drop=True)

    if include_detailed_description:
        detailed_descriptions = {}
        if not df.empty:
            detailed_descriptions = fetch_detailed_descriptions_for_entities(
                df["id"].tolist(),
                language=language,
                headers=DEFAULT_HEADERS,
                timeout=30,
                sentences=detailed_description_sentences,
            )
        df.insert(
            df.columns.get_loc("description") + 1,
            "detailed_description",
            [detailed_descriptions.get(entity_id, "") for entity_id in df["id"]],
        )
        if drop_missing_detailed_description:
            df = df[_series_non_empty_mask(df["detailed_description"])].reset_index(drop=True)

    if df.empty:
        print(f"No search results for: {term!r}")
    return df


def load_wikidata_definition_candidates(
    query_text: str,
    use_detailed_description: bool = True,
    exact_match_text: bool = False,
    filter_name: bool = True,
) -> tuple[pd.DataFrame, str]:
    candidates_df = search_wikidata(
        query_text,
        limit=5,
        exact_match_text=exact_match_text,
        include_detailed_description=use_detailed_description,
        detailed_description_sentences=3,
        drop_missing_detailed_description=use_detailed_description,
        filter_name=filter_name,
    )

    if candidates_df.empty:
        if use_detailed_description:
            raise ValueError(
                f"search_wikidata returned no detailed_description candidates for span={query_text!r}."
            )
        raise ValueError(
            f"search_wikidata returned no description candidates for span={query_text!r}."
        )

    definition_column = "detailed_description" if use_detailed_description else "description"
    candidates_df = candidates_df[_series_non_empty_mask(candidates_df[definition_column])].copy()
    candidates_df = candidates_df.drop_duplicates(subset=["id", definition_column]).reset_index(drop=True)
    if candidates_df.empty:
        raise ValueError(
            f"No usable {definition_column} candidates remained for span={query_text!r}."
        )

    return candidates_df, definition_column


In [6]:
def definition_to_hypothesis(definition: str) -> str:
    cleaned_definition = definition.strip()
    if cleaned_definition.endswith((".", "!", "?")):
        cleaned_definition = cleaned_definition[:-1]
    return f"It refers to {cleaned_definition}."


def extract_cross_encoder_scores(raw_scores, model) -> np.ndarray:
    score_array = np.asarray(raw_scores)
    if score_array.ndim == 1:
        return score_array.astype(float)

    id2label = getattr(model.model.config, "id2label", {}) or {}
    entailment_index = None
    for label_index, label_name in id2label.items():
        if str(label_name).lower() == "entailment":
            entailment_index = int(label_index)
            break

    if entailment_index is None:
        entailment_index = score_array.shape[1] - 1

    return score_array[:, entailment_index].astype(float)


def build_wikidata_candidate_bank(candidates_df: pd.DataFrame, definition_column: str) -> list[dict]:
    candidate_bank = []
    for row in candidates_df.itertuples(index=False):
        definition = str(getattr(row, definition_column)).strip()
        candidate_bank.append(
            {
                "entity_id": row.id,
                "label": row.label,
                "description": row.description,
                "definition_source": definition_column,
                "definition": definition,
                "hypothesis": definition_to_hypothesis(definition),
            }
        )
    return candidate_bank


def evaluate_cross_encoder_on_hotpotqa(
    span_text: str,
    model_name: str = DEFAULT_MODEL_NAME,
    store: dict | None = None,
    kind: str | None = None,
    batch_size: int = DEFAULT_BATCH_SIZE,
    mark_target: bool = False,
    max_records: int | None = None,
    use_detailed_description: bool = True,
    exact_match_text: bool = False,
):
    normalized_span = normalize_text(span_text.strip())
    if not normalized_span:
        raise ValueError("span_text must be a non-empty string.")

    if store is None:
        store = load_hotpot_scan_store()

    records = lookup_records(
        store,
        span_text,
        kind=kind,
        include_text=True,
        include_cleaned_text=True,
    )
    if not records:
        raise ValueError(f"No HotpotQA records were found for span={span_text!r}.")

    if max_records is not None:
        records = records[:max_records]

    candidates_df, definition_column = load_wikidata_definition_candidates(
        span_text.strip(),
        use_detailed_description=use_detailed_description,
        exact_match_text=exact_match_text,
    )
    candidate_bank = build_wikidata_candidate_bank(candidates_df, definition_column=definition_column)
    if not candidate_bank:
        raise ValueError(f"No candidate definitions were built for span={span_text!r}.")

    model = CrossEncoder(model_name)
    evaluation_rows = []

    for record in records:
        prompt_info = build_hotpot_prompt(record, span_text, mark_target=mark_target)
        pairs = [(prompt_info["prompt_text"], candidate["hypothesis"]) for candidate in candidate_bank]
        raw_scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        scores = extract_cross_encoder_scores(raw_scores, model)

        ranked_candidates = sorted(
            [
                {
                    **candidate,
                    "score": float(score),
                }
                for candidate, score in zip(candidate_bank, scores)
            ],
            key=lambda item: item["score"],
            reverse=True,
        )
        top_candidate = ranked_candidates[0]

        evaluation_rows.append(
            {
                "query_span": span_text,
                "normalized_query": normalized_span,
                "title": record["title"],
                "kind": record["kind"],
                "document_idx": record["document_idx"],
                "span": tuple(record["span"]),
                "matched_text": prompt_info["matched_text"],
                "source_text": prompt_info["context_text"],
                "document_text": record["text"],
                "predicted_entity_id": top_candidate["entity_id"],
                "predicted_label": top_candidate["label"],
                "predicted_description": top_candidate["description"],
                "definition_source": top_candidate["definition_source"],
                "predicted_definition": top_candidate["definition"],
                "prediction_score": top_candidate["score"],
                "prompt_text": prompt_info["prompt_text"],
            }
        )

    results_df = pd.DataFrame(evaluation_rows).sort_values(
        ["prediction_score", "title", "document_idx"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    return candidates_df, results_df


def display_hotpot_predictions(results_df: pd.DataFrame, limit: int = 20):
    label_counts_df = (
        results_df.groupby(["predicted_entity_id", "predicted_label", "predicted_description"], dropna=False)
        .size()
        .reset_index(name="sample_count")
        .sort_values(["sample_count", "predicted_entity_id"], ascending=[False, True])
        .reset_index(drop=True)
    )
    total_samples = int(len(results_df))

    print(f"Total samples: {total_samples}")
    display(label_counts_df)

    columns = [
        "title",
        "kind",
        "matched_text",
        "source_text",
        "predicted_label",
        "predicted_definition",
        "prediction_score",
    ]
    display(results_df[columns].head(limit))


In [11]:
pd.set_option("display.max_colwidth", None)

TARGET_SPAN = "director"
QUERY_KIND = None
MARK_TARGET = False
USE_DETAILED_DESCRIPTION = False
EXACT_MATCH_TEXT = True
MAX_RECORDS = 100

candidate_definitions_df, hotpot_results_df = evaluate_cross_encoder_on_hotpotqa(
    span_text=TARGET_SPAN,
    model_name=DEFAULT_MODEL_NAME,
    store=embedding_store,
    kind=QUERY_KIND,
    batch_size=DEFAULT_BATCH_SIZE,
    mark_target=MARK_TARGET,
    max_records=MAX_RECORDS,
    use_detailed_description=USE_DETAILED_DESCRIPTION,
    exact_match_text=EXACT_MATCH_TEXT,
)


In [12]:
definition_column = "detailed_description" if USE_DETAILED_DESCRIPTION else "description"
display(candidate_definitions_df[["id", "label", "description", definition_column]])
display_hotpot_predictions(hotpot_results_df, limit=MAX_RECORDS)
hotpot_results_df.head(100)


,id,label,description,description
0,Q2526255,film director,person who controls the artistic and dramatic aspects of a film production,person who controls the artistic and dramatic aspects of a film production
1,Q3455803,director,director of a creative work,director of a creative work
2,Q3387717,theatrical director,person overseeing the mounting of a theatre production,person overseeing the mounting of a theatre production
3,Q1162163,director,person who leads a particular area of a company or organization,person who leads a particular area of a company or organization
4,Q37293708,Director,family name,family name


Total samples: 100


,predicted_entity_id,predicted_label,predicted_description,sample_count
0,Q3455803,director,director of a creative work,64
1,Q1162163,director,person who leads a particular area of a company or organization,33
2,Q2526255,film director,person who controls the artistic and dramatic aspects of a film production,3


,title,kind,matched_text,source_text,predicted_label,predicted_definition,prediction_score
0,Stargate: Continuum,token,director,"it is directed by Martin Wood, director and producer of many episodes of stargate SG-1 and Stargate Atlantis, written by SG-1 and atlantis creator Brad Wright, and produced by wright and ark of truth director robert c.",director,director of a creative work,3.141982
1,Heidi Ewing,token,director,"other films as a director include The Boys of baraka, freakonomics, and The Education of Mohammed Hussein.",director,director of a creative work,2.148396
2,Kent County Cricket Club in 2010,token,director,it was the first season in charge for director of Cricket Paul Farbrace.,director,director of a creative work,1.527468
3,Independent Spirit Award for Best First Feature,token,director,the first feature designation is applied to the director not the producer(s).,director,director of a creative work,1.515948
4,Kent County Cricket Club in 2011,token,director,it was the second and final season in charge for director of Cricket Paul Farbrace.,director,director of a creative work,1.481409
...,...,...,...,...,...,...,...
95,National Intelligence Estimate,token,director,National Intelligence Estimates (nies) are United States federal government documents that are the authoritative assessment of the director of National Intelligence (DNI) on intelligence related to a particular national security issue.,director,person who leads a particular area of a company or organization,-2.494124
96,National Intelligence Cross,token,director,The National Intelligence Cross is a decoration of the United States Intelligence Community (IC) awarded under the National Intelligence Awards (NIA) program by the office of the director of National Intelligence (ODNI).,director,person who leads a particular area of a company or organization,-2.937858
97,Director of the Defense Intelligence Agency,token,director,"as the chief of the Defense Intelligence Agency, the director is the primary intelligence adviser to the secretary of defense and the chairman of the Joint Chiefs of staff and also answers to the director of National Intelligence through the civilian Under Secretary of defense for intelligence.",director,person who leads a particular area of a company or organization,-2.974411
98,Director of the Defense Intelligence Agency,token,director,"as the chief of the Defense Intelligence Agency, the director is the primary intelligence adviser to the secretary of defense and the chairman of the Joint Chiefs of staff and also answers to the director of National Intelligence through the civilian Under Secretary of defense for intelligence.",director,person who leads a particular area of a company or organization,-2.974411


,query_span,normalized_query,title,kind,document_idx,span,matched_text,source_text,document_text,predicted_entity_id,predicted_label,predicted_description,definition_source,predicted_definition,prediction_score,prompt_text
0,director,director,Stargate: Continuum,token,2638,"(223, 231)",director,"it is directed by Martin Wood, director and producer of many episodes of stargate SG-1 and Stargate Atlantis, written by SG-1 and atlantis creator Brad Wright, and produced by wright and ark of truth director robert c.","Stargate: Continuum is a 2008 Canadian-American military science fiction direct-to-video film in the ""Stargate"" franchise. It is the second sequel to television series ""Stargate SG-1"" following """". It is directed by Martin Wood, director and producer of many episodes of ""Stargate SG-1"" and ""Stargate Atlantis"", written by ""SG-1"" and ""Atlantis"" creator Brad Wright, and produced by Wright and ""Ark of Truth"" director Robert C. Cooper.",Q3455803,director,director of a creative work,description,director of a creative work,3.141982,"Sentence: it is directed by Martin Wood, director and producer of many episodes of stargate SG-1 and Stargate Atlantis, written by SG-1 and atlantis creator Brad Wright, and produced by wright and ark of truth director robert c.\nTarget word: director\n\nQuestion: What does ""director"" mean in this sentence?"
1,director,director,Heidi Ewing,token,323,"(942, 950)",director,"other films as a director include The Boys of baraka, freakonomics, and The Education of Mohammed Hussein.","Heidi Ewing is a director, producer, and writer of documentary films. She and Rachel Grady founded Loki Films in 2001, and have collaborated on several documentaries together. She is best known as the co-director of ""Jesus Camp"", which was nominated for an Academy Award for best documentary in 2006. Next came""12th & Delaware"" (HBO), which premiered at the 2010 Sundance Film Festival. The film ""casts a heart-rending light on the abortion divide"" (LA Times) and was honored with a Peabody Award. ""Detropia"", a poetic look at Ewing's home town, also won several awards, including Best Editing at Sundance 2012, Outstanding Direction and Outstanding Original Score, at the 2013 Cinema Eye Honors for Nonfiction Filmmaking and a News and Documentary Emmy for editing. ""Norman Lear: Just Another Version of You"" was the opening night selection of the 2016 Sundance Film Festival and premiered on PBS American Masters on October 25, 2016. Other films as a director include ""The Boys of Baraka"", ""Freakonomics"", and ""The Education of Mohammed Hussein"".",Q3455803,director,director of a creative work,description,director of a creative work,2.148396,"Sentence: other films as a director include The Boys of baraka, freakonomics, and The Education of Mohammed Hussein.\nTarget word: director\n\nQuestion: What does ""director"" mean in this sentence?"
2,director,director,Kent County Cricket Club in 2010,token,2125,"(344, 352)",director,it was the first season in charge for director of Cricket Paul Farbrace.,"In 2010, Kent County Cricket Club competed in Division One of the County Championship, Group C of the 40-over Clydesdale Bank 40 and the South Group of the Friends Provident t20. Kent also hosted three-day first-class matches at the St Lawrence Ground against Loughborough MCCU and the touring Pakistanis. It was the first season in charge for Director of Cricket Paul Farbrace. The club captain was former England batsman Rob Key who had been club captain since 2006. Kent's overseas players were South African fast bowler Makhaya Ntini until late May, and Sri Lankan leg-spinner Malinga Bandara for the rest of the season.",Q3455803,director,director of a creative work,description,director of a creative work,1.527468,"Sentence: it was the first season in charge for director of Cricket Paul Farbrace.\nTarget word: director\n\nQuestion: What does ""director"" mean in this sentence?"
3,director,director,Independent Spirit